# 🎙️ Somali ASR (Speech-to-Text) — Colab API Server

This notebook loads the Somali wav2vec2 ASR model and serves it as a FastAPI server exposed via `ngrok`.

## Steps:
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to check GPU.
3. Add your `NGROK_TOKEN` and `NGROK_DOMAIN` secrets in the 🔑 key icon on the left sidebar.
   - Get `NGROK_TOKEN` from [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)
   - Get a free static `NGROK_DOMAIN` from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains)
4. Run **Cell 3** to load the ASR model.
5. Run **Cell 4** to start the server and get your public URL.
6. Paste the URL into your backend `.env` as `SOMALI_ASR_URL=<url>`.

In [ ]:
# CELL 1: Install dependencies
# imageio-ffmpeg is pip-installable and ships a static ffmpeg binary — works
# on any environment (Lightning.ai, Colab, Kaggle) without needing sudo/apt.
# The FastAPI server in Cell 4 shells out to it to decode webm/opus uploads.
!pip install -q transformers torch torchaudio fastapi uvicorn pyngrok python-multipart soundfile librosa imageio-ffmpeg
import imageio_ffmpeg
print(f'Dependencies installed! ffmpeg at: {imageio_ffmpeg.get_ffmpeg_exe()}')

In [ ]:
# CELL 2: Verify GPU
import torch

if not torch.cuda.is_available():
    print('WARNING: No GPU found. ASR will run on CPU (slower). Consider enabling GPU.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 3: Load Somali ASR model
import torch
from transformers import AutoModelForCTC, AutoProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ASR_MODEL_ID = 'skydheere/wav2vec2-large-mms-1b-somalia'

print(f'Loading Somali ASR model on {device}...')
asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_ID)
asr_model = AutoModelForCTC.from_pretrained(ASR_MODEL_ID).to(device)
asr_model.eval()
print('Somali ASR model ready!')

In [ ]:
# CELL 4: Start FastAPI server + ngrok tunnel
import os
import subprocess
import tempfile
import numpy as np
import torch
import uvicorn
import imageio_ffmpeg
from threading import Thread
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from pyngrok import ngrok, conf

# ── Secrets ──────────────────────────────────────────────────────────────────
# NGROK_TOKEN and NGROK_DOMAIN come from google.colab.userdata on Colab, or
# from environment variables everywhere else (Lightning.ai, Kaggle, local).
try:
    from google.colab import userdata  # type: ignore
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    NGROK_DOMAIN = userdata.get('NGROK_DOMAIN')
except Exception:
    NGROK_TOKEN = os.environ.get('NGROK_TOKEN')
    NGROK_DOMAIN = os.environ.get('NGROK_DOMAIN')
if not NGROK_TOKEN:
    raise RuntimeError('NGROK_TOKEN is not set — export it as an env var (or Colab secret) before running Cell 4.')
conf.get_default().auth_token = NGROK_TOKEN

app = FastAPI(title='Somali ASR API')

TARGET_SAMPLE_RATE = 16000

# imageio-ffmpeg bundles a static ffmpeg binary and returns its absolute path.
# Using the absolute path avoids "No such file or directory: 'ffmpeg'" errors
# when the uvicorn worker thread doesn't inherit a helpful PATH (the failure
# mode we hit on Lightning.ai where system ffmpeg is not installed and apt
# requires sudo we don't have).
FFMPEG_BIN = imageio_ffmpeg.get_ffmpeg_exe()
print(f'Using ffmpeg at: {FFMPEG_BIN}')

def load_audio_bytes(audio_bytes: bytes, suffix: str = '.webm') -> np.ndarray:
    # Browsers send webm/opus (MediaRecorder). libsndfile can't decode it, and
    # librosa 0.10+ dropped its ffmpeg/audioread fallback — so we shell out to
    # ffmpeg ourselves and read raw 16 kHz mono PCM off stdout.
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(audio_bytes)
        tmp_path = tmp.name
    try:
        proc = subprocess.run(
            [
                FFMPEG_BIN, '-nostdin', '-loglevel', 'error',
                '-i', tmp_path,
                '-f', 's16le', '-acodec', 'pcm_s16le',
                '-ar', str(TARGET_SAMPLE_RATE), '-ac', '1',
                '-',
            ],
            capture_output=True, check=True,
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f'ffmpeg decode failed: {e.stderr.decode(errors="ignore")[:300]}')
    finally:
        os.remove(tmp_path)

    if not proc.stdout:
        raise RuntimeError('ffmpeg produced no audio (empty or corrupt upload?)')
    return np.frombuffer(proc.stdout, dtype=np.int16).astype(np.float32) / 32768.0

def transcribe_audio(audio_array: np.ndarray) -> str:
    inputs = asr_processor(
        audio_array,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = asr_processor.decode(predicted_ids[0])
    return transcription.strip()

@app.get('/health')
def health():
    return {
        'status': 'online',
        'model': 'skydheere/wav2vec2-large-mms-1b-somalia',
        'ffmpeg': FFMPEG_BIN,
    }

@app.post('/transcribe')
async def transcribe(file: UploadFile = File(...)):
    if not file:
        raise HTTPException(status_code=400, detail='No audio file provided')
    try:
        audio_bytes = await file.read()
        suffix = os.path.splitext(file.filename or '')[1] or '.webm'
        audio_array = load_audio_bytes(audio_bytes, suffix=suffix)
        transcription = transcribe_audio(audio_array)
        return JSONResponse({'transcription': transcription, 'language': 'so-SO'})
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Transcription failed: {str(e)}')

# Start uvicorn in background thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='info')

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Connect ngrok tunnel with static domain (if provided)
tunnel = ngrok.connect(8001, 'http', domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(8001, 'http')
public_url = tunnel.public_url

print(f'\n==========================================')
print(f'YOUR ASR URL IS READY!')
print(f'URL: {public_url}')
print(f'==========================================')
print(f'Paste this into your backend .env file:')
print(f'SOMALI_ASR_URL={public_url}')
print(f'==========================================')